## Homework: Agentic RAG

This homework builds a RAG system from scratch and then turns it into an agentic workflow. The knowledge base is the course lesson pages, fetched directly from the course repository.

The repository is organized by module, with each module containing a `lessons/` folder of numbered markdown files. Those lesson pages are the source documents for retrieval in this homework:
- `01-agentic-rag`
- `02-vector-search`
- `03-orchestration`
- `04-evaluation`
- `05-monitoring`
- `06-best-practices`
- `07-project-example`

### Setup

Use the same environment setup as in the lesson’s. One extra library is needed: `gitsource`, which downloads files from GitHub.

For the LLM, this notebook uses `gemini-3.1-flash-lite`.

In [1]:
# Pull the lesson pages from the course repository using a fixed commit

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

print(f"Loaded {len(files)} files")

Loaded 72 files


In [3]:
# Parse each file into a document dictionary with its filename and content.

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

first_doc = files[0].parse()
print(first_doc["filename"])
print(first_doc["content"][:500])

01-agentic-rag/lessons/01-intro.md
# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack


## Q1. How many lesson pages?

How many lesson pages are in the dataset?

- 24
- 72
- 240
- 720

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

answer = len(files)

print(f"Answer: {answer}")

Answer: 72


## Q2. Indexing and searching

Index the documents with minsearch: make `content` a text field and `filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What is the `filename` of the first result?

- `01-agentic-rag/lessons/03-rag.md`
- `01-agentic-rag/lessons/14-agentic-loop.md`
- `04-evaluation/lessons/13-llm-as-judge.md`
- `06-best-practices/lessons/02-hybrid-search.md`

In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=1
)

answer = search_results[0]["filename"]

print(f"Answer: {answer}")

Answer: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

Build a RAG assistant on top of this data.

Use the helper script:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`), `RAGBase` was changed to reflect the new scheme (`filename` and `content`).

Use the index from Q2 and answer:

> How does the agentic loop keep calling the model until it stops?

The question says to use `gpt-5.4-mini` but I have used `gemini-3.1-flash-lite`.

How many input tokens did we send to the model for this request?

- 700
- 7000
- 70000
- 700000

In [25]:
import os # Needed to read the Gemini API key from environment variables
from dotenv import load_dotenv
load_dotenv()

from rag_helper_homework import RAGBase
from openai import OpenAI

openai_client = OpenAI(
    # Fetch the loaded Gemini key
    api_key=os.getenv("GEMINI_API_KEY"),
    
    # Reroute standard OpenAI traffic to Google's servers
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

print(answer.choices[0].message.content)
print(f"Answer: {answer.usage.prompt_tokens} input tokens")

The agentic loop keeps calling the model by wrapping the interaction in a `while` loop that continues to run as long as the model returns function calls.

Here is how the process works:

1.  **Requesting a Call:** The loop sends the current conversation history (which includes instructions, the user prompt, and any previous tool outputs) to the model.
2.  **Checking for Tool Calls:** The code inspects the model's response. If the model returns a `function_call`, the code executes that function using a helper (like `make_call`), appends the tool's output to the conversation history, and sets a flag (`has_function_calls = True`).
3.  **Repeating:** Because the loop is a `while True` construct, it continues to the next iteration, sending the updated message history (now including the new tool results) back to the model.
4.  **Stopping:** The loop stops when the model finishes its reasoning and returns a final response that does *not* contain any function calls. At this point, the `has_fun

## Q4. Chunking

The lesson pages are long, and some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page.

A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

Use:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000`:
- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks.
- Since `step` is smaller than `size`, consecutive chunks overlap by `size - step` characters.
- Every chunk keeps the original fields (`filename`) and adds `start` and `content`.

How many chunks do you get?

- 70
- 295
- 1100
- 4500

In [19]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"{chunks[0]['filename']}\n{chunks[1]['filename']}")

answer = len(chunks)

print(f"Answer: {answer}")

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/01-intro.md
Answer: 295


## Q5. RAG with chunking

Chunking makes each request smaller because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4, point your RAG at the chunk index, and answer the same query again. Read the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

- about the same
- 3× fewer
- 10× fewer
- 30× fewer

In [26]:
from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

assistant = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
)

chunk_answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

print(chunk_answer.choices[0].message.content)
print(f"{chunk_answer.usage.prompt_tokens} input tokens")

answer = round(answer.usage.prompt_tokens / chunk_answer.usage.prompt_tokens)

print(f"Answer: {answer}× fewer")

The agentic loop uses a `while True` loop to continuously call the model. The process works as follows:

1. **Model Reasoning:** The model evaluates the current conversation history (messages) and decides whether to return a final answer or call a tool.
2. **Execution:** The code iterates through the model's response. If the model specifies a `function_call`, the code executes that function, adds the result to the message history, and sets a flag (`has_function_calls = True`).
3. **Loop Continuation:** If `has_function_calls` is true, the loop continues to the next iteration, providing the model with the tool's output so it can reason about the next step.
4. **Exit Condition:** The loop stops when the model returns a response without any function calls. In the code, this is handled by checking the `has_function_calls` flag; if it remains `False` after processing the model's response, the loop terminates using a `break` statement.
2615 input tokens
Answer: 3× fewer


## Q6. Turning it into an agent

Give the LLM a `search` tool and let it decide when and what to search.

You can use `toyaikit`, the small agent library from the module, or anything else you prefer.

Create a `search` function that uses the chunk index. Give it a type hint and a docstring.

Build an agent with your `search` tool and run it. Use these instructions for the agent:

> You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many times it called the `search` tool.

How many times did the agent call `search`?

- 0
- 4
- 10
- 20

In [49]:
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback

# Create an OpenAI-compatible client for Gemini
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Raw OpenAI-compatible SDK client, pointed at Gemini
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL_ID = os.getenv("MODEL_ID")

# ToyAIKit wrapper around the client
llm_client = OpenAIChatCompletionsClient(
    model=MODEL_ID,
    client=openai_client
)

print("ToyAIKit and Gemini client configured successfully.")

ToyAIKit and Gemini client configured successfully.


In [50]:
# Define the search tool
def search(query: str) -> list[dict]:
    """
    Search the chunk index for lesson content relevant to the query.
    """
    return index.search(
        query,
        num_results=5
    )

# Register the search function directly, ToyAIKit can generate the schema automatically
agent_tools = Tools()
agent_tools.add_tool(search)

print("Search tool defined and registered with automatically generated schema.")

Search tool defined and registered with automatically generated schema.


In [51]:
# Create the chat interface and runner for the notebook and insert instructions
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.",
    chat_interface=chat_interface,
    llm_client=llm_client
)

# Run prompt and count tool use ("Function call")
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


Feature,Plain RAG,Agentic Loop
Control,Fixed (Hardcoded sequence),Dynamic (LLM decides next step)
Flexibility,Low; runs once regardless of result,"High; can retry, rephrase, or use multiple tools"
Recovery,None; fails if search is poor,"Can ""self-correct"" by trying different actions"
Complexity,"Simple, predictable, fast","More complex, non-deterministic, costlier"
